<a href="https://colab.research.google.com/github/sandeep-sc11/stock-retrun/blob/main/optimum_portfolio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Installing requierd libraries
!pip install yfinance cufflinks plotly numpy pandas

In [ ]:
# Import core libraries
import pandas as pd
import numpy as np
import yfinance as yf

# Import visualization libraries
import cufflinks as cf
import plotly.io as pio

# Date utilities
from datetime import date
from dateutil.relativedelta import relativedelta

# Configure plotly for Google Colab
cf.go_offline()
pio.renderers.default = "colab"

print("All libraries imported successfully ")

In [ ]:
stocks = [
    'HDFCBANK.NS','INFY.NS','ITC.NS','SUNPHARMA.NS',
    'RELIANCE.NS','ICICIBANK.NS','TCS.NS','LT.NS',
    'SBIN.NS','BHARTIARTL.NS','AXISBANK.NS'
]

market_index = '^NSEI'

end_date = date.today()
start_date = end_date - relativedelta(years=3)

stock_data = yf.download(stocks, start=start_date, end=end_date, auto_adjust=False, progress=False)['Close']
market_data = yf.download(market_index, start=start_date, end=end_date, auto_adjust=False, progress=False)['Close']

# Drop missing rows
stock_data = stock_data.dropna()

# Align using common dates
common_dates = stock_data.index.intersection(market_data.index)

stock_data = stock_data.loc[common_dates]
market_data = market_data.loc[common_dates]

print("Stock Data Shape:", stock_data.shape)
print("Market Data Shape:", market_data.shape)

In [ ]:
stock_data.isnull().sum()

In [ ]:
stock_data.head()

In [ ]:
stock_data.tail()

In [ ]:
stock_data.columns

In [ ]:
stock_data = stock_data.dropna()

In [ ]:
# Calculate daily returns

stock_returns = stock_data.pct_change().dropna()
market_returns = market_data.pct_change().dropna()

# Align again to ensure perfect matching
common_dates = stock_returns.index.intersection(market_returns.index)

stock_returns = stock_returns.loc[common_dates]
market_returns = market_returns.loc[common_dates]

print("Stock Returns Shape:", stock_returns.shape)
print("Market Returns Shape:", market_returns.shape)

stock_returns.head()

In [ ]:
# Calculate Beta for each stock

betas = {}

market_variance = market_returns.var().values[0]

for stock in stock_returns.columns:
    covariance = stock_returns[stock].cov(market_returns.iloc[:,0])
    beta = covariance / market_variance
    betas[stock] = beta

beta_df = pd.DataFrame.from_dict(betas, orient='index', columns=['Beta'])

beta_df.sort_values(by='Beta', ascending=False)

In [ ]:
# Risk-free rate (daily)
risk_free_daily = 0.0742 / 252

# Average daily market return
avg_market_return = market_returns.mean().values[0]

alphas = {}

for stock in stock_returns.columns:

    # Average daily stock return
    avg_stock_return = stock_returns[stock].mean()

    # Beta value (from previous calculation)
    beta_value = beta_df.loc[stock, 'Beta']

    # CAPM expected return
    expected_return = risk_free_daily + beta_value * (avg_market_return - risk_free_daily)

    # Alpha calculation
    alpha = avg_stock_return - expected_return

    alphas[stock] = alpha

alpha_df = pd.DataFrame.from_dict(alphas, orient='index', columns=['Daily Alpha'])

alpha_df.sort_values(by='Daily Alpha', ascending=False)

In [ ]:
# Annualized risk-free rate
risk_free_annual = 0.0742

# Annualized market return
annual_market_return = market_returns.mean().values[0] * 252

comparison = {}

for stock in stock_returns.columns:

    # Annualized actual return
    annual_stock_return = stock_returns[stock].mean() * 252

    # Beta
    beta_value = beta_df.loc[stock, 'Beta']

    # CAPM expected annual return
    expected_return_annual = risk_free_annual + beta_value * (annual_market_return - risk_free_annual)

    # Annualized alpha
    alpha_annual = annual_stock_return - expected_return_annual

    comparison[stock] = [
        beta_value,
        annual_stock_return,
        expected_return_annual,
        alpha_annual
    ]

comparison_df = pd.DataFrame.from_dict(
    comparison,
    orient='index',
    columns=['Beta', 'Actual Return (Annual)', 'CAPM Expected (Annual)', 'Alpha (Annual)']
)

comparison_df.sort_values(by='Alpha (Annual)', ascending=False)

In [ ]:
# Security Market Line Visualization

import matplotlib.pyplot as plt
import numpy as np

# Risk-free and market return (annual)
risk_free = 0.0742
market_return = annual_market_return

# Generate beta values for SML line
beta_range = np.linspace(0, 1.5, 100)
sml_line = risk_free + beta_range * (market_return - risk_free)

plt.figure(figsize=(10,6))

# Plot SML
plt.plot(beta_range, sml_line, label="Security Market Line")

# Plot actual stocks
for stock in comparison_df.index:
    beta_val = comparison_df.loc[stock, 'Beta']
    actual_ret = comparison_df.loc[stock, 'Actual Return (Annual)']
    plt.scatter(beta_val, actual_ret)
    plt.text(beta_val, actual_ret, stock)

plt.xlabel("Beta (Systematic Risk)")
plt.ylabel("Annual Return")
plt.title("Security Market Line (CAPM)")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# Portfolio Statistics Setup

import numpy as np
import pandas as pd

risk_free = 0.0742

mean_returns = stock_returns.mean() * 252
cov_matrix = stock_returns.cov() * 252

num_assets = len(mean_returns)

In [ ]:
print(results.shape) #correct

In [ ]:
# Identify Optimal Portfolios

max_sharpe_idx = np.argmax(results[2])
min_vol_idx = np.argmin(results[0])

max_sharpe_vol = results[0, max_sharpe_idx]
max_sharpe_ret = results[1, max_sharpe_idx]

min_vol_vol = results[0, min_vol_idx]
min_vol_ret = results[1, min_vol_idx]

print("Max Sharpe Return:", max_sharpe_ret)
print("Max Sharpe Volatility:", max_sharpe_vol)

print("\nMin Volatility Return:", min_vol_ret)
print("Min Volatility Volatility:", min_vol_vol)

In [ ]:
# Efficient Frontier with Capital Market Line

cml_x = np.linspace(0, results[0,:].max(), 100)

cml_y = risk_free + ((max_sharpe_ret - risk_free) / max_sharpe_vol) * cml_x

plt.figure(figsize=(10,6))

plt.scatter(results[0,:], results[1,:],
            c=results[2,:], cmap='viridis')

plt.plot(cml_x, cml_y, color='black', linewidth=2, label='Capital Market Line')

plt.scatter(max_sharpe_vol, max_sharpe_ret,
            marker='*', color='red', s=200,
            label='Max Sharpe')

plt.scatter(min_vol_vol, min_vol_ret,
            marker='*', color='green', s=200,
            label='Min Volatility')

plt.xlabel('Annualized Volatility')
plt.ylabel('Annualized Return')
plt.title('Efficient Frontier with Capital Market Line')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# Equal Weight Portfolio

equal_weights = np.ones(num_assets) / num_assets

equal_return = np.dot(equal_weights, mean_returns)

equal_volatility = np.sqrt(
    np.dot(equal_weights.T, np.dot(cov_matrix, equal_weights))
)

equal_sharpe = (equal_return - risk_free) / equal_volatility

print("Equal Weight Portfolio:")
print("Return:", equal_return)
print("Volatility:", equal_volatility)
print("Sharpe Ratio:", equal_sharpe)

In [ ]:
comparison_table = pd.DataFrame({
    'Portfolio': ['Max Sharpe', 'Minimum Variance', 'Equal Weight'],
    'Return': [max_sharpe_ret, min_vol_ret, equal_return],
    'Volatility': [max_sharpe_vol, min_vol_vol, equal_volatility],
    'Sharpe Ratio': [
        (max_sharpe_ret - risk_free) / max_sharpe_vol,
        (min_vol_ret - risk_free) / min_vol_vol,
        equal_sharpe
    ]
})

comparison_table

In [ ]:
# Extract Optimal Portfolio Weights

max_sharpe_weights = weights_record[max_sharpe_idx]
min_vol_weights = weights_record[min_vol_idx]

max_sharpe_portfolio = pd.DataFrame({
    'Stock': stock_returns.columns,
    'Weight (Max Sharpe)': max_sharpe_weights
}).sort_values(by='Weight (Max Sharpe)', ascending=False)

min_vol_portfolio = pd.DataFrame({
    'Stock': stock_returns.columns,
    'Weight (Min Vol)': min_vol_weights
}).sort_values(by='Weight (Min Vol)', ascending=False)

print("Max Sharpe Portfolio Allocation:\n")
print(max_sharpe_portfolio)

print("\nMinimum Volatility Portfolio Allocation:\n")
print(min_vol_portfolio)

In [ ]:
# Plot Max Sharpe Portfolio Allocation

import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
plt.bar(max_sharpe_portfolio['Stock'],
        max_sharpe_portfolio['Weight (Max Sharpe)'])

plt.xticks(rotation=90)
plt.title("Max Sharpe Portfolio Allocation")
plt.ylabel("Weight")
plt.grid(True)
plt.show()

In [ ]:
# Plot Min Volatility Portfolio Allocation

plt.figure(figsize=(10,6))
plt.bar(min_vol_portfolio['Stock'],
        min_vol_portfolio['Weight (Min Vol)'])

plt.xticks(rotation=90)
plt.title("Minimum Volatility Portfolio Allocation")
plt.ylabel("Weight")
plt.grid(True)
plt.show()